# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd
import numpy as np

In [2]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [3]:
# explorar datasets
orders.info()
display(orders.head())
display(orders.describe())
print(orders.isna().sum())
print('filas duplicadas:', orders.duplicated().sum() )
print('id_pedidos repetidos:', orders['id_pedido'].duplicated().sum())
for col in ['pais', 'dispositivo', 'fuente_referencia', 'categoria_producto']:
    print(orders[col].value_counts(dropna=False))
    print()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64
filas duplicadas: 100
id_pedidos repetidos: 100
Colombia     7520
Mexico       7502
Argentina    7291
mexico        865
colombia      823
argentina     799
NaN           300
Name: pais, dtype: int64

desktop    12759
mobile     12321
NaN           20
Name: dispositivo, dtype: int64

social         8428
organic        8329
paid_search    8313
NaN              30
Name: fuente_referencia, dtype: int64

Hogar          8385
Moda           8323
Electronica    8312
NaN              80
Name: categoria_producto, dtype: int64



In [4]:
catalog.info()
display(catalog.head())
display(catalog.describe())
print(catalog.isna().sum())
print('filas duplicadas', catalog.duplicated().sum())
print('Productos repetidos:', catalog['nombre_producto'].duplicated().sum())

for col in ['categoria_producto', 'proveedor']:
    print(catalog[col].value_counts(dropna=False))
    print()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


nombre_producto       0
categoria_producto    0
costo_unitario        0
proveedor             0
dtype: int64
filas duplicadas 0
Productos repetidos: 0
Electrónica    3
Moda           2
Hogar          2
Name: categoria_producto, dtype: int64

Greene-Smith               1
Fuller, Pena and Myers     1
Rivera, Carr and Finley    1
Mcmillan-Rhodes            1
King Ltd                   1
Bowers LLC                 1
Long-Reid                  1
Name: proveedor, dtype: int64



In [5]:


marketing.info()
display(marketing.head())
display(marketing.describe())
print(marketing.isna().sum())
print()
print('Filas duplicadas:', marketing.duplicated().sum())
for col in ['pais', 'canal']:
    print(marketing[col].value_counts(dropna=False))
    print()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000


fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64

Filas duplicadas: 0
Argentina    540
Mexico       540
Colombia     540
Name: pais, dtype: int64

paid_search    507
social         506
organic        506
NaN            101
Name: canal, dtype: int64



---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [6]:
print('nulos antes:', marketing['fecha'].isna().sum())
marketing['fecha']= pd.to_datetime(marketing['fecha'], errors="coerce")
print('nulos despues:', marketing['fecha'].isna().sum())
marketing.info()

nulos antes: 0
nulos despues: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1519 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


In [7]:
print('nulos antes:', orders['fecha_hora_pedido'].isna().sum())
orders['fecha_hora_pedido']= pd.to_datetime(orders['fecha_hora_pedido'], errors= "coerce")
print('nulos despues:', orders['fecha_hora_pedido'].isna().sum())
orders.info()



nulos antes: 0
nulos despues: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25100 non-null  object        
 1   id_usuario          25100 non-null  object        
 2   fecha_hora_pedido   25100 non-null  datetime64[ns]
 3   pais                24800 non-null  object        
 4   dispositivo         25080 non-null  object        
 5   fuente_referencia   25070 non-null  object        
 6   nombre_producto     25070 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25100 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.3+ MB


fecha_hora_pedido se convirtió a datetime64 sin generar nulos: las 25.100 fechas eran válidas.

In [8]:
sin_cantidad = orders[orders['cantidad'].isna()]
print(sin_cantidad[['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']].isna().sum())
print()
print(orders['cantidad'].isna().groupby(orders['pais']).mean())

cantidad           50
precio_unitario    50
monto_descuento    50
monto_total         0
dtype: int64

pais
Argentina    0.002743
Colombia     0.001862
Mexico       0.001733
argentina    0.001252
colombia     0.001215
mexico       0.001156
Name: cantidad, dtype: float64


Los nulos de cantidad, precio_unitario y monto_descuento están en las mismas 50 filas (0,2%). No se pueden reconstruir porque faltan 3 datos de la fórmula, y están repartidos parejo entre países (MCAR). Se eliminan.

In [9]:
print('Filas antes:', len(orders))
orders = orders.dropna(subset=['cantidad'])
print('Filas después:', len(orders))
print(orders[['cantidad', 'precio_unitario', 'monto_descuento']].isna().sum())


Filas antes: 25100
Filas después: 25050
cantidad           0
precio_unitario    0
monto_descuento    0
dtype: int64


Se eliminaron 50 filas. orders quedó con 25.050 pedidos y sin nulos en cantidad, precio_unitario ni monto_descuento.

In [10]:
for col in ['pais', 'dispositivo', 'fuente_referencia', 'categoria_producto']:
    orders[col] = orders[col].str.strip().str.title()
    print(orders[col].value_counts(dropna=False))
    print()

orders['nombre_producto'] = orders['nombre_producto'].str.strip()

Mexico       8353
Colombia     8328
Argentina    8069
NaN           300
Name: pais, dtype: int64

Desktop    12733
Mobile     12297
NaN           20
Name: dispositivo, dtype: int64

Social         8412
Organic        8311
Paid_Search    8297
NaN              30
Name: fuente_referencia, dtype: int64

Hogar          8385
Moda           8323
Electronica    8312
NaN              30
Name: categoria_producto, dtype: int64



Se estandarizaron pais, dispositivo, fuente_referencia y categoria_producto con .str.strip().str.title(), porque había el mismo valor escrito con mayúsculas y minúsculas (ej. "Argentina" y "argentina"). Resultado: 3 países, 2 dispositivos, 3 fuentes y 3 categorías, sin variantes.

In [11]:
cols = ['pais', 'dispositivo', 'fuente_referencia', 'categoria_producto', 'nombre_producto']
print(orders[cols].isna().sum())

orders = orders.dropna(subset=['nombre_producto'])

for col in ['pais', 'dispositivo', 'fuente_referencia', 'categoria_producto']:
    orders[col] = orders[col].fillna('Desconocido')

print(orders[cols].isna().sum())
print('Filas:', len(orders))

pais                  300
dispositivo            20
fuente_referencia      30
categoria_producto     30
nombre_producto        30
dtype: int64
pais                  0
dispositivo           0
fuente_referencia     0
categoria_producto    0
nombre_producto       0
dtype: int64
Filas: 25020


Nulos: pais (300), fuente_referencia (30), categoria_producto (30), nombre_producto (30), dispositivo (20). Se eliminaron las 30 filas sin nombre_producto, porque sin producto no se puede asignar costo en el Paso 2. El resto se rellenó con "Desconocido": son columnas descriptivas y esos pedidos tienen revenue real. Filas: 25.020.

In [12]:
print('Filas duplicadas completas:', orders.duplicated().sum())
orders = orders.drop_duplicates()
print('Filas:', len(orders))

print('id_pedido repetidos:', orders['id_pedido'].duplicated().sum())
print(orders[orders['id_pedido'].duplicated(keep=False)].sort_values('id_pedido').head(10))

Filas duplicadas completas: 100
Filas: 24920
id_pedido repetidos: 0
Empty DataFrame
Columns: [id_pedido, id_usuario, fecha_hora_pedido, pais, dispositivo, fuente_referencia, nombre_producto, categoria_producto, cantidad, precio_unitario, monto_descuento, monto_total]
Index: []


Había 100 filas duplicadas exactas, que inflaban el revenue y el número de pedidos. Se eliminaron y no quedó ningún id_pedido repetido. Filas: 24.920.

In [13]:
print('cantidad <= 0:', (orders['cantidad'] <= 0).sum())
print('precio_unitario <= 0:', (orders['precio_unitario'] <= 0).sum())
print('monto_descuento < 0:', (orders['monto_descuento'] < 0).sum())
print('monto_total <= 0:', (orders['monto_total'] <= 0).sum())

orders = orders[(orders['cantidad'] > 0) & (orders['precio_unitario'] > 0)]
print('Filas:', len(orders))

cantidad <= 0: 4
precio_unitario <= 0: 0
monto_descuento < 0: 0
monto_total <= 0: 4
Filas: 24916


Había 4 pedidos con cantidad y monto_total negativos (ej. cantidad −2, total −192,62). Probablemente son devoluciones o reembolsos, no ventas. Se excluyeron porque restarían revenue y distorsionarían el ticket y la cantidad promedio. Filas: 24.916.

In [14]:

esperado = orders['cantidad'] * orders['precio_unitario'] - orders['monto_descuento']
diferencia = (orders['monto_total'] - esperado).abs()
print('Pedidos con diferencia > 1:', (diferencia > 1).sum())
print(orders[diferencia > 1][['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']].head())


Pedidos con diferencia > 1: 0
Empty DataFrame
Columns: [cantidad, precio_unitario, monto_descuento, monto_total]
Index: []


Se verificó que monto_total = cantidad × precio_unitario − monto_descuento, con tolerancia de 1 por redondeo. Las 3 inconsistencias eran devoluciones ya excluidas; después de la limpieza quedan 0.

In [15]:
print('cantidad con decimales:', (orders['cantidad'] % 1 != 0).sum())
orders['cantidad'] = orders['cantidad'].astype(int)
orders.info()

cantidad con decimales: 0
<class 'pandas.core.frame.DataFrame'>
Int64Index: 24916 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24916 non-null  object        
 1   id_usuario          24916 non-null  object        
 2   fecha_hora_pedido   24916 non-null  datetime64[ns]
 3   pais                24916 non-null  object        
 4   dispositivo         24916 non-null  object        
 5   fuente_referencia   24916 non-null  object        
 6   nombre_producto     24916 non-null  object        
 7   categoria_producto  24916 non-null  object        
 8   cantidad            24916 non-null  int64         
 9   precio_unitario     24916 non-null  float64       
 10  monto_descuento     24916 non-null  float64       
 11  monto_total         24916 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(7)
memory usage: 2.5+ MB


cantidad era float por los NaN, pero no tenía decimales reales; se convirtió a int. orders queda con 24.916 pedidos, 0 nulos y tipos correctos.

In [16]:
print(orders['cantidad'].describe())
print(orders.sort_values('cantidad', ascending=False)[['id_pedido', 'nombre_producto', 'cantidad', 'precio_unitario', 'monto_total']].head(10))

count    24916.000000
mean         7.123214
std        297.072369
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max      20000.000000
Name: cantidad, dtype: float64
       id_pedido     nombre_producto  cantidad  precio_unitario  monto_total
3668  order_3668  Laptop-Gaming-16GB     20000           348.31    6966200.0
3656  order_3656  Laptop-Gaming-16GB     20000           297.66    5953200.0
3722  order_3722  Laptop-Gaming-16GB     20000           442.01    8840200.0
3726  order_3726  Laptop-Gaming-16GB     20000           290.85    5817000.0
3643  order_3643  Laptop-Gaming-16GB     10000           238.15    2381500.0
3586  order_3586  Laptop-Gaming-16GB     10000           490.35    4903500.0
3522  order_3522  Laptop-Gaming-16GB     10000           280.55    2805500.0
3521  order_3521  Laptop-Gaming-16GB     10000            43.14     431400.0
3689  order_3689  Laptop-Gaming-16GB     10000            87.69     876900.0
3748  order_3748  Lapto

In [17]:
print('Filas antes:', len(orders))
orders = orders[orders['cantidad'] < 100]
print('Filas después:', len(orders))
print(orders['cantidad'].value_counts().sort_index())

Filas antes: 24916
Filas después: 24906
1    12330
2    12576
Name: cantidad, dtype: int64


Había 10 pedidos de Laptop-Gaming-16GB con 10.000 o 20.000 unidades (el resto de pedidos tiene 1 o 2). Juntos sumaban ≈42 millones, cerca del 80% del revenue total, así que se trataron como error de datos y se eliminaron. Filas: 24.906.

In [18]:
print(orders['cantidad'].value_counts().sort_index())

1    12330
2    12576
Name: cantidad, dtype: int64


In [19]:
catalog['nombre_producto'] = catalog['nombre_producto'].str.strip()
catalog['categoria_producto'] = catalog['categoria_producto'].str.strip().str.title()
catalog['categoria_producto'] = catalog['categoria_producto'].str.replace('Electrónica', 'Electronica')
print(catalog['categoria_producto'].value_counts(dropna=False))
print(catalog.isna().sum())

Electronica    3
Moda           2
Hogar          2
Name: categoria_producto, dtype: int64
nombre_producto       0
categoria_producto    0
costo_unitario        0
proveedor             0
dtype: int64


catalog no tiene nulos. Se estandarizó el texto y "Electrónica" se cambió a "Electronica" para que coincida con orders.

In [20]:
print('Filas duplicadas completas:', catalog.duplicated().sum())
catalog = catalog.drop_duplicates()
print('Productos repetidos:', catalog['nombre_producto'].duplicated().sum())
print(catalog[catalog['nombre_producto'].duplicated(keep=False)].sort_values('nombre_producto'))

Filas duplicadas completas: 0
Productos repetidos: 0
Empty DataFrame
Columns: [nombre_producto, categoria_producto, costo_unitario, proveedor]
Index: []


In [21]:
print('costo_unitario <= 0:', (catalog['costo_unitario'] <= 0).sum())
print(catalog['costo_unitario'].describe())

costo_unitario <= 0: 0
count      7.000000
mean     102.252857
std      111.011563
min       10.120000
25%       16.905000
50%       25.210000
75%      182.975000
max      280.680000
Name: costo_unitario, dtype: float64


In [22]:
faltan = ~orders['nombre_producto'].isin(catalog['nombre_producto'])
print('Pedidos sin producto en catalog:', faltan.sum())
print(orders[faltan]['nombre_producto'].unique())

Pedidos sin producto en catalog: 0
[]


In [23]:
canal_desde_id = marketing['id_campaña'].str.rsplit('_', n=1).str[0]
marketing['canal'] = marketing['canal'].fillna(canal_desde_id)

for col in ['pais', 'canal']:
    marketing[col] = marketing[col].str.strip().str.title()
    print(marketing[col].value_counts(dropna=False))
    print()
print(marketing.isna().sum())

Argentina    540
Mexico       540
Colombia     540
Name: pais, dtype: int64

Social         540
Paid_Search    540
Organic        540
Name: canal, dtype: int64

fecha         0
pais          0
id_campaña    0
canal         0
gasto         0
dtype: int64


canal tenía 101 nulos, pero el canal viene escrito dentro de id_campaña (ej. paid_search_Mexico). Se reconstruyó desde ahí en lugar de borrar filas. Resultado: 0 nulos y 540 filas por canal y por país.

In [24]:
print('Filas duplicadas completas:', marketing.duplicated().sum())
marketing = marketing.drop_duplicates()
print('Filas:', len(marketing))
print('gasto <= 0:', (marketing['gasto'] <= 0).sum())
print(marketing['gasto'].describe())

Filas duplicadas completas: 0
Filas: 1620
gasto <= 0: 0
count    1620.00000
mean     1772.74292
std       734.43294
min       501.11000
25%      1128.03000
50%      1782.42500
75%      2420.68500
max      2999.36000
Name: gasto, dtype: float64


marketing no tiene filas duplicadas (1.620 filas). El gasto va de 501 a 2.999, sin ceros, negativos ni valores extremos.

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

Resumen de la limpieza. orders pasó de 25.100 a 24.906 pedidos (−194, el 0,8%): 50 sin cantidad, 30 sin producto, 100 duplicados, 4 devoluciones y 10 pedidos con cantidades imposibles. catalog (7 productos) y marketing (1.620 filas) quedaron sin nulos ni duplicados.

In [25]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [26]:
# 1. Traer el costo de cada producto desde catalog
df = orders.merge(catalog[['nombre_producto', 'costo_unitario']], on='nombre_producto', how='left')
print('Filas orders:', len(orders), '| Filas df:', len(df))
print('Pedidos sin costo:', df['costo_unitario'].isna().sum())

# 2. Costo de cada pedido
df['costo'] = df['cantidad'] * df['costo_unitario']

# 3. Revisar que pedidos y marketing cubran el mismo periodo
print('Periodo orders:', orders['fecha_hora_pedido'].min(), 'a', orders['fecha_hora_pedido'].max())
print('Periodo marketing:', marketing['fecha'].min(), 'a', marketing['fecha'].max())

# 4. KPIs
revenue = df['monto_total'].sum()
costo_total = df['costo'].sum()
gasto_marketing = marketing['gasto'].sum()
profit = revenue - costo_total - gasto_marketing

print('Revenue:', round(revenue, 2))
print('Costo total:', round(costo_total, 2))
print('Gasto marketing:', round(gasto_marketing, 2))
print('Profit:', round(profit, 2))
print('Margen %:', round(profit / revenue * 100, 2))


Filas orders: 24906 | Filas df: 24906
Pedidos sin costo: 0
Periodo orders: 2025-01-01 00:00:00 a 2025-06-30 00:00:00
Periodo marketing: 2025-01-01 00:00:00 a 2025-06-29 00:00:00
Revenue: 9610018.94
Costo total: 3828869.01
Gasto marketing: 2871843.53
Profit: 2909306.4
Margen %: 30.27


Rentabilidad (enero–junio 2025). Todos los pedidos encontraron su costo en catalog y no se duplicaron filas (24.906). Pedidos y marketing cubren el mismo periodo, así que el gasto se resta completo.

Revenue: 9.610.019
Costo de productos: 3.828.869
Gasto en marketing: 2.871.844
Profit: 2.909.306 (margen 30,3%)

El negocio es rentable. Sin marketing el margen sería 60%; el marketing consume casi la mitad de esa ganancia, por lo que es la principal palanca para mejorar el profit.

In [27]:
# 1. Ticket promedio: cuánto paga en promedio cada pedido (cada fila = 1 pedido)
print('Ticket promedio:', round(df['monto_total'].mean(), 2))

# 2. Cantidad promedio de productos por pedido
print('Cantidad promedio por pedido:', round(df['cantidad'].mean(), 2))
print()

# 3. Producto más vendido: unidades por producto, de mayor a menor
print(df.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False))
print()

# 4. Gasto de marketing por canal, de mayor a menor
print(marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False))

Ticket promedio: 385.85
Cantidad promedio por pedido: 1.5

nombre_producto
Vacuum-Pro-Black        6284
Blender-XL-Red          6279
Jacket-Winter-M         6256
Sneakers-Urban-42       6172
Laptop-Gaming-16GB      4198
Tablet-Standard-64GB    4153
Phone-Pro-128GB         4140
Name: cantidad, dtype: int64

canal
Social         976818.37
Organic        972650.96
Paid_Search    922374.20
Name: gasto, dtype: float64


Comportamiento de ventas.

Ticket promedio: 385,85 por pedido.
Cantidad promedio: 1,5 productos por pedido (todos los pedidos son de 1 o 2 unidades).
Producto más vendido: Vacuum-Pro-Black (6.284 unidades), prácticamente empatado con Blender-XL-Red (6.279) y Jacket-Winter-M (6.256). Los productos de Hogar y Moda venden ~6.200 unidades cada uno; los de Electrónica, ~4.150.
Marketing por canal: Social 976.818, Organic 972.651, Paid_Search 922.374, casi un tercio cada uno. Llama la atención que Organic tenga gasto, porque el tráfico orgánico suele ser gratis; conviene confirmar con el equipo qué incluye.

In [28]:
# 1. Ganancia por pedido y pedidos vendidos por debajo del costo
df['ganancia'] = df['monto_total'] - df['costo']
print('Pedidos vendidos por debajo del costo:', (df['ganancia'] < 0).sum())
print()

# 2. Ganancia y margen por producto (sin marketing: no está asignado a productos)
por_producto = df.groupby('nombre_producto')[['monto_total', 'costo', 'ganancia']].sum()
por_producto['margen_%'] = por_producto['ganancia'] / por_producto['monto_total'] * 100
print(por_producto.sort_values('ganancia', ascending=False).round(1))
print()

# 3. Revenue por cada peso de marketing, por canal
por_canal = pd.DataFrame({
    'revenue': df.groupby('fuente_referencia')['monto_total'].sum(),
    'gasto_marketing': marketing.groupby('canal')['gasto'].sum()
})
por_canal['revenue_por_peso'] = por_canal['revenue'] / por_canal['gasto_marketing']
print(por_canal.round(2))

Pedidos vendidos por debajo del costo: 4528

                      monto_total      costo   ganancia  margen_%
nombre_producto                                                  
Vacuum-Pro-Black        1621051.9   104314.4  1516737.4      93.6
Sneakers-Urban-42       1566075.9   106220.1  1459855.8      93.2
Phone-Pro-128GB         1057134.6    41896.8  1015237.8      96.0
Tablet-Standard-64GB    1062922.8   104697.1   958225.7      90.2
Blender-XL-Red          1611574.6  1109122.6   502452.0      31.2
Jacket-Winter-M         1606586.2  1184323.4   422262.8      26.3
Laptop-Gaming-16GB      1084673.0  1178294.6   -93621.7      -8.6

                revenue  gasto_marketing  revenue_por_peso
Organic      3178228.55        972650.96              3.27
Paid_Search  3209280.07        922374.20              3.48
Social       3222510.32        976818.37              3.30


Segmentos rentables.

4.528 pedidos (18%) se vendieron por debajo del costo del producto.
Laptop-Gaming-16GB genera pérdidas: −93.622 (margen −8,6%). Su costo unitario es 280,68 y se vende en promedio a ~258.
Jacket-Winter-M (26%) y Blender-XL-Red (31%) tienen márgenes bajos por su costo alto.
Vacuum-Pro-Black, Sneakers-Urban-42, Phone-Pro-128GB y Tablet-Standard-64GB dejan más del 90% de margen. Vacuum es el más vendido y el más rentable (1,52M de ganancia).
Por canal, cada peso de marketing genera entre 3,27 y 3,48 de revenue. Paid_Search es el más eficiente (3,48), aunque la diferencia entre canales es pequeña. Es una aproximación: se asigna el revenue de cada fuente_referencia al gasto del canal con el mismo nombre.

RappiPlus es rentable: 2,9M de profit sobre 9,6M de revenue (margen 30%). Pero esa ganancia viene de los productos de bajo costo: Laptop-Gaming-16GB se vende por debajo del costo, y Jacket y Blender dejan poco margen.

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [29]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión a la base de datos
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'YOUR_PASSWORD',  # credentials provided by TripleTen, removed for security
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [30]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [31]:
query_eventos = '''
SELECT nombre_evento,
       COUNT(*) AS total_eventos,
       COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY usuarios_unicos DESC;
'''
pd.read_sql(query_eventos, con=engine)

,nombre_evento,total_eventos,usuarios_unicos
0,first_visit,29957,7796
1,add_to_cart,24157,7634
2,select_item,23887,7582
3,begin_checkout,17971,7208
4,add_payment_info,12018,6250
5,purchase,12010,6240


In [32]:
# PARTE 1: Totales del funnel
query_totals = '''
SELECT CASE nombre_evento
           WHEN 'first_visit' THEN 1
           WHEN 'select_item' THEN 2
           WHEN 'add_to_cart' THEN 3
           WHEN 'begin_checkout' THEN 4
           WHEN 'add_payment_info' THEN 5
           WHEN 'purchase' THEN 6
       END AS paso,
       nombre_evento,
       COUNT(DISTINCT id_usuario) AS usuarios
FROM events
GROUP BY nombre_evento
ORDER BY paso;
'''
totals = pd.read_sql(query_totals, con=engine)
totals

,paso,nombre_evento,usuarios
0,1,first_visit,7796
1,2,select_item,7582
2,3,add_to_cart,7634
3,4,begin_checkout,7208
4,5,add_payment_info,6250
5,6,purchase,6240


Usuarios únicos por etapa, ordenados según el flujo real del usuario (no por cantidad): first_visit (7.796) → select_item (7.582) → add_to_cart (7.634) → begin_checkout (7.208) → add_payment_info (6.250) → purchase (6.240). Hay 52 usuarios más en add_to_cart que en select_item: algunos agregan al carrito sin un evento select_item previo (posible acceso directo desde listados o un vacío en el tracking).

In [33]:
# PARTE 2: Conversiones
query_conversion = '''
WITH funnel AS (
    SELECT CASE nombre_evento
               WHEN 'first_visit' THEN 1
               WHEN 'select_item' THEN 2
               WHEN 'add_to_cart' THEN 3
               WHEN 'begin_checkout' THEN 4
               WHEN 'add_payment_info' THEN 5
               WHEN 'purchase' THEN 6
           END AS paso,
           nombre_evento,
           COUNT(DISTINCT id_usuario) AS usuarios
    FROM events
    GROUP BY nombre_evento
)
SELECT paso,
       nombre_evento,
       usuarios,
       LAG(usuarios) OVER (ORDER BY paso) - usuarios AS usuarios_perdidos,
       ROUND(100.0 * usuarios / LAG(usuarios) OVER (ORDER BY paso), 2) AS conv_paso_anterior,
       ROUND(100.0 * usuarios / FIRST_VALUE(usuarios) OVER (ORDER BY paso), 2) AS conv_desde_inicio
FROM funnel
ORDER BY paso;
'''
conversion = pd.read_sql(query_conversion, con=engine)
conversion

,paso,nombre_evento,usuarios,usuarios_perdidos,conv_paso_anterior,conv_desde_inicio
0,1,first_visit,7796,NaN,NaN,100.00
1,2,select_item,7582,214.0,97.26,97.26
2,3,add_to_cart,7634,-52.0,100.69,97.92
3,4,begin_checkout,7208,426.0,94.42,92.46
4,5,add_payment_info,6250,958.0,86.71,80.17
5,6,purchase,6240,10.0,99.84,80.04


Conversión del funnel.

Conversión final: 80,04% de los usuarios que visitan la app compran.
Mayor drop-off: de begin_checkout a add_payment_info se pierden 958 usuarios (conversión 86,71%). Es el punto donde más usuarios abandonan.
Segundo drop-off: de add_to_cart a begin_checkout se pierden 426 usuarios (94,42%).
El paso final funciona: el 99,84% de quienes ingresan sus datos de pago completan la compra.

Recomendación: priorizar la pantalla de datos de pago del checkout: simplificar el formulario, ofrecer más métodos de pago y guardar tarjetas. Recuperar la mitad de esos 958 usuarios sumaría ~480 compradores más. El test A/B del Paso 5 evalúa justamente un cambio en el checkout.

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [34]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [35]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [36]:
# ¿Qué valores tiene dias_despues_registro y cuántos usuarios están activos en cada uno?
query_dias = '''
SELECT dias_despues_registro,
       COUNT(*) AS filas,
       SUM(activo) AS activos
FROM user_activity
GROUP BY dias_despues_registro
ORDER BY dias_despues_registro;
'''
display(pd.read_sql(query_dias, con=engine))

# ¿Desde cuándo hasta cuándo se registraron usuarios y cuántos son?
query_registro = '''
SELECT MIN(CAST(fecha_registro AS DATE)) AS primera,
       MAX(CAST(fecha_registro AS DATE)) AS ultima,
       COUNT(*) AS filas,
       COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM users;
'''
display(pd.read_sql(query_registro, con=engine))

,dias_despues_registro,filas,activos
0,7,8000,3360.0
1,14,8000,3355.0
2,21,8000,3350.0
3,28,8000,3250.0


,primera,ultima,filas,usuarios_unicos
0,2025-01-01,2025-05-31,8000,8000


user_activity tiene una fila por usuario en los días 7, 14, 21 y 28 después del registro (8.000 filas por día). Se define semana 1 = día 7, semana 2 = día 14 y semana 3 = día 21, y solo cuentan las filas con activo = 1. users tiene 8.000 usuarios únicos registrados entre enero y mayo de 2025, lo que da 5 cohortes mensuales.

In [37]:
# Retención por cohortes
query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT id_usuario,
           DATE_TRUNC('month', CAST(fecha_registro AS DATE))::DATE AS cohorte
    FROM users
),
retencion AS (
    SELECT c.cohorte,
           COUNT(DISTINCT c.id_usuario) AS usuarios_iniciales,
           COUNT(DISTINCT CASE WHEN a.dias_despues_registro = 7  AND a.activo = 1 THEN a.id_usuario END) AS retenido_w1,
           COUNT(DISTINCT CASE WHEN a.dias_despues_registro = 14 AND a.activo = 1 THEN a.id_usuario END) AS retenido_w2,
           COUNT(DISTINCT CASE WHEN a.dias_despues_registro = 21 AND a.activo = 1 THEN a.id_usuario END) AS retenido_w3
    FROM cohortes c
    LEFT JOIN user_activity a ON c.id_usuario = a.id_usuario
    GROUP BY c.cohorte
)
SELECT cohorte,
       usuarios_iniciales,
       retenido_w1,
       retenido_w2,
       retenido_w3,
       ROUND(100.0 * retenido_w1 / usuarios_iniciales, 2) AS semana_1,
       ROUND(100.0 * retenido_w2 / usuarios_iniciales, 2) AS semana_2,
       ROUND(100.0 * retenido_w3 / usuarios_iniciales, 2) AS semana_3
FROM retencion
ORDER BY cohorte;
'''

cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01,1687,695,676,706,41.20,40.07,41.85


Retención por cohortes. Las 5 cohortes (enero–mayo 2025, 8.000 usuarios) retienen entre 40% y 44% en las semanas 1, 2 y 3, sin diferencias importantes entre meses.

La mayor pérdida ocurre en los primeros 7 días: cerca del 58% de los usuarios ya no está activo en la semana 1.
Después de la semana 1 la retención se estabiliza (~42%): quien supera la primera semana tiende a quedarse.
Algunas cohortes suben de una semana a otra (ej. febrero: 42,3% → 44,0%) porque hay usuarios que vuelven después de una semana inactivos.

Recomendación: concentrar esfuerzos en la primera semana: onboarding, notificaciones y beneficios de RappiPlus en los primeros días para que el usuario haga su segunda compra. Mejorar la retención de la semana 1 tiene más impacto que trabajar las semanas siguientes.

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

**Hipótesis estadística**
- **H₀ (Hipótesis nula):** la tasa de conversión del grupo control es igual a la del grupo tratamiento; el nuevo checkout no cambia la conversión.
- **H₁ (Hipótesis alternativa):** la tasa de conversión de control y tratamiento es diferente.

**Test estadístico:** z-test de dos proporciones, porque la métrica es binaria (convirtió 1 / no convirtió 0) y se comparan dos grupos independientes.
**Nivel de significancia alpha:** 0,05

In [38]:

# Cargar el experimento
exp = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
exp.info()
display(exp.head())

# Calidad de datos
print('Filas duplicadas:', exp.duplicated().sum())
print('Usuarios repetidos:', exp['id_usuario'].duplicated().sum())
print(exp['variante'].value_counts(dropna=False))
print(exp['convirtio'].value_counts(dropna=False))

# Usuarios que aparecen en las dos variantes
variantes_por_usuario = exp.groupby('id_usuario')['variante'].nunique()
print('Usuarios en ambas variantes:', (variantes_por_usuario > 1).sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


Filas duplicadas: 0
Usuarios repetidos: 0
tratamiento    5035
control        4965
Name: variante, dtype: int64
0    8401
1    1599
Name: convirtio, dtype: int64
Usuarios en ambas variantes: 0


El experimento tiene 10.000 usuarios, sin nulos, duplicados ni usuarios asignados a ambas variantes. Los grupos están balanceados: tratamiento 5.035 (50,35%) y control 4.965 (49,65%). Conversión global: 15,99% (1.599 de 10.000).

In [39]:
from statsmodels.stats.proportion import proportions_ztest

# 1. Conversión por grupo
resumen = exp.groupby('variante')['convirtio'].agg(['sum', 'count', 'mean'])
resumen.columns = ['conversiones', 'usuarios', 'tasa_conversion']
print(resumen)
print()

# 2. Diferencia relativa (lift): cuánto más (o menos) convierte tratamiento vs control
tasa_control = resumen.loc['control', 'tasa_conversion']
tasa_trat = resumen.loc['tratamiento', 'tasa_conversion']
lift = (tasa_trat - tasa_control) / tasa_control * 100
print('Lift:', round(lift, 2), '%')

# 3. z-test de dos proporciones
conversiones = [resumen.loc['control', 'conversiones'], resumen.loc['tratamiento', 'conversiones']]
usuarios = [resumen.loc['control', 'usuarios'], resumen.loc['tratamiento', 'usuarios']]
z_stat, p_value = proportions_ztest(conversiones, usuarios)
print('z:', round(z_stat, 3))
print('p-value:', round(p_value, 4))

alpha = 0.05
if p_value < alpha:
    print('Se rechaza H0: la diferencia en conversión es estadísticamente significativa.')
else:
    print('No se rechaza H0: no hay evidencia de que el nuevo checkout cambie la conversión.')

             conversiones  usuarios  tasa_conversion
variante                                            
control               779      4965         0.156898
tratamiento           820      5035         0.162860

Lift: 3.8 %
z: -0.813
p-value: 0.4161
No se rechaza H0: no hay evidencia de que el nuevo checkout cambie la conversión.


Resultado del test A/B.

Control: 15,69% de conversión (779 de 4.965).
Tratamiento: 16,29% (820 de 5.035).
Diferencia: +0,6 puntos porcentuales (lift de +3,8%).
z-test de dos proporciones: p-value = 0,416 > alpha 0,05, por lo que no se rechaza H₀.

La diferencia observada no es estadísticamente significativa y puede deberse al azar. No hay evidencia de que el nuevo checkout aumente la conversión.

Recomendación. No lanzar el nuevo checkout esperando un aumento de conversión, porque el resultado no lo respalda. Con ~5.000 usuarios por grupo, el test solo puede detectar diferencias de unos 2 puntos; confirmar una diferencia de 0,6 puntos requeriría cerca de 60.000 usuarios por grupo. Como la mayor fuga del funnel (Paso 3) está en el ingreso de datos de pago, se recomienda diseñar el próximo test enfocado en ese paso: tarjetas guardadas, más métodos de pago y menos campos.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard


**Dashboard en Tableau Public:** https://public.tableau.com/views/RappiPlusDashboard_17916413859700/RappiPlusOverviewEjecutivoenerojunio2025

El dashboard tiene dos vistas: **Overview** (KPIs, tendencia mensual, revenue YTD y revenue/profit por producto) y **Detalle** (tabla por pedido con profit en verde/rojo, filtros de fecha y categoría, y clic en un producto para ver sus pedidos).